In [ ]:
from model_arch import *

@dataclass
class NERConfig(LMConfig):
    num_classes: int = 25

class DecoderForTokenClassification(nn.Module):
    def __init__(self, cfg: NERConfig, pretrained_lm: DecoderLM):
        super().__init__()
        self.num_classes = cfg.num_classes
        self.transformer = pretrained_lm
        self.transformer.head = nn.Identity()
        self.classifier = nn.Linear(cfg.d_model, cfg.num_classes)

    def forward(self, idx):
        hidden_states = self.transformer(idx)
        logits = self.classifier(hidden_states)
        
        return logits

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer, r = 8, alpha = 8.0):
        super().__init__()
        self.base = base_layer
        self.scale = alpha / r
        
        for param in self.base.parameters():
            param.requires_grad = False
            
        in_dim = base_layer.in_features 
        out_dim = base_layer.out_features
        
        self.lora_A = nn.Linear(in_dim, r, bias=False)
        self.lora_B = nn.Linear(r, out_dim, bias=False)
        
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_B(self.lora_A(x))

In [ ]:
import pickle
from torch.utils.data import Dataset, DataLoader

with open("label_mappings.pkl", "rb") as f:
    a = pickle.load(f)
    id2label = a['id2label']
    label2id = a['label2id']

with open("integer_chunked_ner_folds.pkl", "rb") as f:
    all_folds = pickle.load(f)

train_chunks = []
for i in range(4):
    train_chunks.extend(all_folds[i])

val_chunks = all_folds[4]

class FastNERDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long)
        }

train_loader = DataLoader(FastNERDataset(train_chunks), batch_size=8, shuffle=True)
val_loader = DataLoader(FastNERDataset(val_chunks), batch_size=8, shuffle=False)

In [ ]:
from torch import optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_cfg = LMConfig(vocab_size=50260)
pretrained_model = DecoderLM(base_cfg)
pretrained_model.load_state_dict(torch.load("dec.pt", map_location=device))
ner_cfg = NERConfig(vocab_size=50260, num_classes=25)
ner_model = DecoderForTokenClassification(ner_cfg, pretrained_model)

rank = 8
alpha_val = 8.0

for block in ner_model.transformer.blocks:
    block.attn.q_proj = LoRALinear(block.attn.q_proj, r=rank, alpha=alpha_val)
    block.attn.v_proj = LoRALinear(block.attn.v_proj, r=rank, alpha=alpha_val)

for name, param in ner_model.named_parameters():
    if "lora_" in name or "classifier" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

ner_model.to(device)
trainable_params = [p for p in ner_model.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=4e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

In [ ]:
for epoch in range(10):
    ner_model.train()
    total_train_loss = 0
    
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        logits = ner_model(input_ids) 
        loss = loss_fn(logits.view(-1, ner_model.num_classes), labels.view(-1))
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    torch.save(ner_model.state_dict(), "checkpoint.pt")
        
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f}")